# Transactions — 01: Single STAC Item Ingest

**Persona:** Data Engineer

**Goal:** Ingest a single Sentinel-2 STAC item into an existing collection via the STAC
Transactions API, verify the item was stored, and confirm the collection item count increased.

---

## ⚠️ Active Bug — LayerConfig Precondition

Items may receive a `201 Created` response but a subsequent GET may return 0 items if the
target collection does not yet have a physical **LayerConfig** assigned. The ingestion
pipeline requires a LayerConfig to know which storage driver to write to.

**Run `catalog/02_collection_layer_config.ipynb` first** to ensure the collection is
properly configured before executing this notebook.

---

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- A catalog identified by `CATALOG_ID` already exists
- A collection identified by `COLLECTION_ID` already exists inside that catalog
- The collection has a LayerConfig (physical storage driver assigned)
- `DYNASTORE_WRITE_TOKEN` is set if the instance requires authentication

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_WRITE_TOKEN` | _(empty)_ | Bearer token for write operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `sentinel2-l2a` | Target collection |

In [ ]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL    = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
WRITE_TOKEN = os.environ.get("DYNASTORE_WRITE_TOKEN", "")
CATALOG_ID  = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "sentinel2-l2a")

headers = {"Authorization": f"Bearer {WRITE_TOKEN}"} if WRITE_TOKEN else {}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30)

print(f"Base URL     : {BASE_URL}")
print(f"Catalog      : {CATALOG_ID}")
print(f"Collection   : {COLLECTION_ID}")
print(f"Auth header  : {'set' if WRITE_TOKEN else 'not set'}")

## Step 1 — Prepare the STAC item

Build a minimal but complete Sentinel-2 L2A STAC item covering a tile in East Africa
(UTM zone 37, roughly Ethiopia/South Sudan). All required STAC 1.0.0 fields are present:
`type`, `stac_version`, `id`, `geometry`, `bbox`, `properties.datetime`, `assets`, `links`.

In [ ]:
import json
import datetime

ITEM_ID = "S2A_MSIL2A_2024-01-15"

item = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": ITEM_ID,
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [33.0, 3.0],
                [36.0, 3.0],
                [36.0, 6.0],
                [33.0, 6.0],
                [33.0, 3.0]
            ]
        ]
    },
    "bbox": [33.0, 3.0, 36.0, 6.0],
    "properties": {
        "datetime": "2024-01-15T10:30:00Z",
        "platform": "sentinel-2a",
        "eo:cloud_cover": 12.5,
        "external_id": "S2A_MSIL2A_2024-01-15_T37MBN"
    },
    "assets": {},
    "links": []
}

print(json.dumps(item, indent=2))

## Step 2 — Ingest the item

POST the item to the STAC Transactions endpoint. A `201 Created` response confirms the
platform accepted the record. The response body echoes the stored item with any
server-side additions (e.g. `geoid`, generated links).

In [ ]:
resp = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=item,
)
assert resp.status_code == 201, (
    f"Expected 201 Created, got {resp.status_code}: {resp.text[:400]}"
)

created = resp.json()
item_id = created.get("id", ITEM_ID)
print(f"Item ingested — id: {item_id}")
print(json.dumps(created, indent=2))

In [ ]:
# Verify persistence: GET the individual item back
resp = client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{item_id}"
)
assert resp.status_code == 200, (
    f"Expected 200 OK, got {resp.status_code}: {resp.text[:400]}"
)

fetched = resp.json()
assert fetched["id"] == item_id, (
    f"ID mismatch: expected {item_id!r}, got {fetched['id']!r}"
)
print(f"GET verified — id: {fetched['id']}")
print(f"  platform      : {fetched['properties'].get('platform')}")
print(f"  cloud_cover   : {fetched['properties'].get('eo:cloud_cover')}")
print(f"  datetime      : {fetched['properties'].get('datetime')}")

## Step 3 — Verify collection item count increased

List all items in the collection and confirm at least one result is returned.
The `context.returned` field in the response carries the count of records in this page.

In [ ]:
resp = client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"limit": 5},
)
assert resp.status_code == 200, (
    f"Expected 200 OK, got {resp.status_code}: {resp.text[:400]}"
)

result = resp.json()
returned = result.get("context", {}).get("returned", len(result.get("features", [])))
assert returned >= 1, f"Expected at least 1 item, context.returned={returned}"

print(f"Items in collection (this page): {returned}")
print(f"numberMatched: {result.get('numberMatched', 'n/a')}")
for feat in result.get("features", []):
    print(f"  - {feat['id']}")

## Edge cases

### Case A — POST without geometry

The platform may accept items with a `null` geometry. DynaStore will auto-assign a
platform-generated `geoid` based on the `external_id` hash when no geometry is provided.
The item is still stored but spatial queries will return no results for it.

In [ ]:
item_no_geom = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": "S2A_MSIL2A_NO_GEOM",
    "geometry": None,
    "bbox": None,
    "properties": {
        "datetime": "2024-01-16T08:00:00Z",
        "platform": "sentinel-2a",
        "external_id": "S2A_MSIL2A_NO_GEOM_TEST"
    },
    "assets": {},
    "links": []
}

resp = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=item_no_geom,
)
# The platform accepts null geometry and stores the item (201).
# A 422 here would indicate the endpoint enforces geometry as required.
print(f"POST without geometry → {resp.status_code}")
if resp.status_code == 201:
    body = resp.json()
    print(f"  Stored geometry : {body.get('geometry')}")
    # Clean up this edge-case item
    client.delete(
        f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{item_no_geom['id']}"
    )
elif resp.status_code == 422:
    print("  Platform requires geometry — null not accepted (expected on strict deployments)")
else:
    print(f"  Response body: {resp.text[:300]}")

### Case B — POST with non-EPSG:4326 geometry

STAC requires WGS84 (EPSG:4326) coordinates. If a client erroneously submits coordinates
in a projected SRS (e.g. UTM metres), PostGIS will attempt to transform them.
The operation typically returns `200` or `201` but the stored geometry will be meaningless
unless the caller also passes the correct `source_srid`. Document actual behaviour below.

In [ ]:
# UTM Zone 37N coordinates (metres, not degrees) submitted without srid hint.
# Expected: 200/201 — PostGIS accepts the values and treats them as EPSG:4326,
# which places the point in the middle of the ocean. No validation error.
item_utm = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": "S2A_MSIL2A_UTM_TEST",
    "geometry": {
        "type": "Point",
        # These look like UTM metres but are treated as lon/lat by the API
        "coordinates": [500000.0, 400000.0]
    },
    "bbox": [500000.0, 400000.0, 500000.0, 400000.0],
    "properties": {
        "datetime": "2024-01-17T09:00:00Z",
        "platform": "sentinel-2a",
        "external_id": "S2A_MSIL2A_UTM_TEST"
    },
    "assets": {},
    "links": []
}

resp = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=item_utm,
)
# PostGIS will store whatever coordinates it receives when no re-projection is triggered.
# Behaviour: 201 accepted, coordinates stored as-is (semantically wrong but not rejected).
print(f"POST with projected coordinates → {resp.status_code}")
print("  Note: coordinates are stored as-is; no automatic re-projection without source_srid.")
if resp.status_code in (200, 201):
    stored = resp.json().get("geometry", {})
    print(f"  Stored geometry: {stored}")
    # Clean up
    client.delete(
        f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{item_utm['id']}"
    )
else:
    print(f"  Response: {resp.text[:300]}")

## Teardown

Remove the item ingested in Step 2 to leave the collection in its original state.

In [ ]:
resp = client.delete(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{item_id}"
)
assert resp.status_code == 204, (
    f"Expected 204 No Content, got {resp.status_code}: {resp.text[:400]}"
)
print(f"Deleted item {item_id!r} — status {resp.status_code}")

# Confirm the item is gone
resp_check = client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{item_id}"
)
assert resp_check.status_code == 404, (
    f"Expected 404 after deletion, got {resp_check.status_code}"
)
print(f"Confirmed: GET {item_id!r} → 404 Not Found")

client.close()